# C10-competition-craft — Practice p17 — Solution

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260804
SIGNAL = ["honey_stores_kg", "autumn_hive_mass_kg", "varroa_mite_index",
          "forager_traffic_per_min", "brood_frames", "daily_temp_swing_c",
          "queen_age_years"]
sweep_ks = np.array([5, 7, 9, 11, 15])

df = pd.read_csv("../data/train.csv")
FEATURES = [c for c in df.columns if c != "outcome"]
X = df[FEATURES]
y = df["outcome"].to_numpy()
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=150, random_state=SEED, stratify=y
)


def val_f1_for(features, k):
    candidate = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=int(k))),
    ]).fit(X_tr[features], y_tr)
    return float(f1_score(y_val, candidate.predict(X_val[features]), average="macro"))


current_features = FEATURES
current_k = 5
current_score = val_f1_for(current_features, current_k)
log = [{"step": "baseline", "change": "scaled 5-NN, all 12 features",
        "val_f1": current_score, "accepted": True}]

iter1_scores = np.array([val_f1_for(current_features, k) for k in sweep_ks])
iter1_k = int(sweep_ks[np.argmax(iter1_scores)])
current_k = iter1_k
current_score = float(iter1_scores.max())
log.append({"step": "iter-1", "change": f"sweep k -> {current_k}",
            "val_f1": current_score, "accepted": True})

iter2_score = val_f1_for(SIGNAL, current_k)
iter2_accepted = bool(iter2_score > current_score)
if iter2_accepted:
    current_features = SIGNAL
    current_score = iter2_score
log.append({"step": "iter-2", "change": "12 features -> 7 SIGNAL columns",
            "val_f1": iter2_score, "accepted": iter2_accepted})

iter3_scores = np.array([val_f1_for(current_features, k) for k in sweep_ks])
iter3_k = int(sweep_ks[np.argmax(iter3_scores)])
iter3_score = float(iter3_scores.max())
iter3_changed = bool(iter3_k != current_k)
if iter3_changed:
    current_k = iter3_k
    current_score = iter3_score
log.append({"step": "iter-3", "change": f"re-sweep k -> {iter3_k} (unchanged)",
            "val_f1": iter3_score, "accepted": iter3_changed})

log_df = pd.DataFrame(log, columns=["step", "change", "val_f1", "accepted"])
final_val_f1 = float(current_score)
final_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=current_k)),
]).fit(X[current_features], y)


def predict_labels(X_test):
    return pd.Series(final_pipe.predict(X_test[current_features]), index=X_test.index)


probe = X.iloc[150:190]
probe_out = predict_labels(probe)
contract_ok = bool(
    isinstance(probe_out, pd.Series)
    and len(probe_out) == len(probe)
    and probe_out.index.equals(probe.index)
    and set(probe_out.unique()) <= set(np.unique(y))
)

self_scores = {"W-A1": 1, "W-A2": 1, "W-B1": 1,
               "W-B2": 1, "W-C1": 1, "W-C2": 1}
self_total = int(sum(self_scores.values()))
log_df

### Writeup

**Approach.** I submitted a `StandardScaler` + `KNeighborsClassifier(n_neighbors=11)` pipeline on the 7 `SIGNAL` features. I froze a stratified 150-row validation carve at `random_state=20260804`; the selected recipe scored validation macro-F1 0.819876, then I refit it on all 600 labeled rows.

**Intuition.** kNN votes by distance, so scaling prevents large-scale columns from dominating, while removing five near-noise dimensions makes neighborhood similarity reflect the seven colony-vigor signals. Macro-F1 matches the roughly 2:1 imbalance by giving the struggling minority equal class-level weight.

**Alternatives.** The all-feature scaled 5-NN baseline scored 0.766582; the first (k)-sweep chose (k=11) at 0.810348, and switching to `SIGNAL` improved that to 0.819876. The final re-sweep again chose (k=11), so it made no state change. Limitation: three validation-selected moves have worn this one carve, so 0.819876 is likely optimistic for new rows.

### Answer check

In [ ]:
assert list(log_df.columns) == ["step", "change", "val_f1", "accepted"]
assert log_df["step"].tolist() == ["baseline", "iter-1", "iter-2", "iter-3"]
assert log_df.shape == (4, 4)
assert log_df["accepted"].dtype == bool
assert log_df["accepted"].tolist() == [True, True, True, False]
expected = np.array([0.7665823769694612, 0.8103481812876873,
                     0.8198760747394207, 0.8198760747394207])
assert np.allclose(log_df["val_f1"].to_numpy(), expected, atol=1e-12, rtol=0)
assert current_k == 11 and current_features == SIGNAL
assert np.isclose(final_val_f1, 0.8198760747394207, atol=1e-12, rtol=0)
assert contract_ok is True
assert set(self_scores) == {"W-A1", "W-A2", "W-B1", "W-B2", "W-C1", "W-C2"}
assert self_total == 6